In [1]:
from pathlib import Path  # Import Path for file system path operations and management
import numpy as np  # Import NumPy for numerical computations and array operations
import pandas as pd  # Import Pandas for data manipulation and analysis with DataFrames
import matplotlib.pyplot as plt  # Import Matplotlib for creating static, interactive visualizations
import seaborn as sns  # Import Seaborn for statistical data visualization built on Matplotlib

from sklearn.model_selection import train_test_split  # Import function to split dataset into training and testing subsets
from sklearn.metrics import accuracy_score, classification_report  # Import function to calculate various metric
import tensorflow as tf

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from helper import fn_plot_confusion_matrix,fn_plot_tf_hist, fn_plot_torch_hist

C:\Users\PGCP-AI\AppData\Local\anaconda3\envs\dnn\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


ModuleNotFoundError: No module named 'helper'

In [ ]:
RANDOM_STATE = 24 # for initialization ----- REMEMBER: to remove at the time of promotion to production
rng = np.random.default_rng(seed = RANDOM_STATE) # Set Random Seed for reproducible  results
np.random.seed(RANDOM_STATE)

tf.random.set_seed(RANDOM_STATE)

EPOCHS = 100 # number of cycles to run
ALPHA = 0.001 # learning rate
BATCH_SIZE = 64
TEST_SIZE = 0.2


# Set parameters for decoration of plots
params = {'legend.fontsize' : 'large',
          'figure.figsize'  : (9,9),
          'axes.labelsize'  : 'x-large',
          'axes.titlesize'  :'x-large',
          'xtick.labelsize' :'large',
          'ytick.labelsize' :'large',
         }
PATIENCE = 10
LR_FACTOR = 0.1
LR_PATIENCE = 5

plt.rcParams.update(params) # update rcParams
CMAP = plt.cm.coolwarm
plt.style.use('seaborn-v0_8-darkgrid') # plt.style.use('ggplot')

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

print(tf.__version__, physical_devices)

In [ ]:
train_df = pd.read_csv('fashion-mnist_train.csv', header = 0)
test_df = pd.read_csv('fashion-mnist_test.csv', header = 0)

In [ ]:
class_names = {0: 'T-shirt/top',1:'Trouser',2:'Pullover',3:'Dress',4:'Coat',
               5:'Sandal', 6: 'Shirt',7: 'Sneaker', 8:'Bag', 9: 'Ankle boot'}

In [ ]:
def split_feature_label(row):
  feature = tf.reshape(row[1:], [28, 28, 1])
  label = row[0]
  return feature, label

In [ ]:
class FashionMNISTDataset(Dataset):

    def __init__(self, dataframe, transform = None):

        super(FashionMNISTDataset, self).__init__()
        self.labels = dataframe.iloc[:, 0].to_numpy().astype(np.long)
        self.images = dataframe.iloc[:, 1:].to_numpy().astype(np.float32)
        self.transform = transform


    def __len__(self):
        return len(self.labels)


    def __getitem__(self, index):
        image = self.images[index].reshape(28,28,1)
        image = image/255.0
        label = self.labels[index]

        # convert to tensor, (C, H, W)
        image = torch.from_numpy(image).permute(2, 0, 1)

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [ ]:
train_dataset = FashionMNISTDataset(train_df)
test_dataset = FashionMNISTDataset(test_df)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

len(train_loader), len(test_loader)

In [ ]:
def show_samples(data_loader, class_labels, num_samples = BATCH_SIZE):
  images, labels = next(iter(data_loader))

  cols = 8
  rows = num_samples//8
  fig, axes = plt.subplots(rows, cols, figsize = (15, rows*2))
  axes = axes.ravel()

  for i in range(num_samples):
    img = images[i].squeeze().numpy()
    ax = axes[i]
    ax.imshow(img, cmap = plt.cm.binary)
    ax.set_title(class_labels[labels[i].item()])
    ax.axis('off')

  plt.tight_layout()
  plt.show()


In [ ]:
show_samples(train_loader, class_names)

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self, num_classes = 10):
        super(FashionCNN, self).__init__()
        dor1 = 0.1
        dor2 = 0.2
        dor3 = 0.3
        dor4 = 0.4

        # Input channel 1, output channels 32, kernel size 3x3
        in_channels1 = 1
        out_channels1 = 32
        self.conv1 = nn.Conv2d(in_channels1, out_channels1, kernel_size=3, padding='same')

        self.bn1 = nn.BatchNorm2d(out_channels1)
        self.rlu1 = nn.ReLU()
        self.maxpool1 = nn.MaxPool2d(kernel_size=(2,2), stride=(2,2))
        self.do1 = nn.Dropout(dor1)

        out_channels2 = 64

        self.conv2 = nn.Conv2d(out_channels1, out_channels2, kernel_size=3)
        self.bn2 = nn.BatchNorm2d(out_channels2)
        self.rlu2 = nn.ReLU()
        self.maxpool2 = nn.MaxPool2d(kernel_size=(2,2), stride=(2,2))
        self.do2 = nn.Dropout(dor2)

        out_channels3 = 128

        self.conv3 = nn.Conv2d(out_channels2, out_channels3, kernel_size=3)
        self.bn3 = nn.BatchNorm2d(out_channels3)
        self.rlu3 = nn.ReLU()
        self.maxpool3 = nn.MaxPool2d(kernel_size=(2,2), stride=(2,2))
        self.do3 = nn.Dropout(dor3)
        self.fc_input_size = 128*4*4

        self.fc1 = nn.Linear(self.fc_input_size, 128)
        self.bn4 = nn.BatchNorm1d(128)
        self.rlu4 = nn.ReLU()
        self.do4 = nn.Dropout(dor4)

        self.fc2 = nn.Linear(128, num_classes)


    def forward(self, x):
        x = self.do1(self.maxpool1(self.rlu1(self.bn1(self.conv1(x)))))
        x = self.do2(self.maxpool2(self.rlu2(self.bn2(self.conv2(x)))))
        x = self.do3(self.rlu3(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1)
        x = self.do4(self.rlu4(self.bn4(self.fc1(x))))
        x = self.fc2(x)
        return x

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = FashionCNN(num_classes=10).to(device)
print(model)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),
                             lr=ALPHA,
                             )

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                     mode='min',
                                                     factor=LR_FACTOR,
                                                     patience=LR_PATIENCE
                                                     )

In [ ]:
def train_model(model, train_loader, optimizer = optimizer, loss_fn = loss_fn, scheduler = scheduler, device = device):

    model.train()
    total_loss = 0
    total_acc = 0
    for train_X, train_y in train_loader:

        train_X, train_y = train_X.to(device), train_y.to(device)

        optimizer.zero_grad()
        outputs = model(train_X)

        batch_loss = loss_fn(outputs, train_y)

        batch_loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += batch_loss.item()

        _, y_pred = torch.max(outputs, 1)
        batch_acc = accuracy_score(train_y.cpu(), y_pred.cpu())

        total_acc += batch_acc

    avg_loss = total_loss / len(train_loader)
    avg_acc = total_acc / len(train_loader)

    return avg_loss, avg_acc

In [ ]:
def evaluate(model, test_loader, loss_fn = loss_fn, device = device):

    model.eval()
    total_loss = 0
    total_acc = 0

    with torch.inference_mode():

        for test_X, test_y in test_loader:

            test_X, test_y = test_X.to(device), test_y.to(device)

            outputs = model(test_X)

            batch_loss = loss_fn(outputs, test_y)
            total_loss += batch_loss.item()


            _, y_pred = torch.max(outputs, 1)
            batch_acc = accuracy_score(test_y.cpu(), y_pred.cpu())

            total_acc += batch_acc


    avg_loss = total_loss / len(test_loader)
    avg_acc = total_acc / len(test_loader)

    return avg_loss, avg_acc

In [ ]:
history = {'epoch':[], 'train_loss':[], 'test_loss':[], 'train_acc': [], 'test_acc': []}

best_test_loss = float('inf')
best_model_state = None

for epoch in range(EPOCHS):

    train_loss, train_acc = train_model(model, train_loader)

    test_loss, test_acc = evaluate(model, test_loader)

    scheduler.step(test_loss)

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_model_state = model.state_dict().copy()

    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['train_acc'].append(train_acc)
    history['test_acc'].append(test_acc)

    if epoch % 10 == 0:
      # fmtStr = | Loss: {:.5f} / {:.5f} | Acc: {:.5f} / {:.5f} |
      print(f'Epoch:{epoch:5d}/{EPOCHS:5d} | '
            f'LR:{optimizer.param_groups[0]['lr']:0.6f} | '
            f'Train Loss:{train_loss:0.4f} | '
            f'Test Loss:{test_loss:0.4f} | '
            )
print(f'Best Test Loss:{best_test_loss:0.4f}')

if best_model_state is not None:
    model.load_state_dict(best_model_state)

In [ ]:
loss_df = pd.DataFrame(history)

In [ ]:
fn_plot_torch_hist(loss_df)

In [ ]:
num_samples=50

cols = 10
rows = num_samples // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 2))
axes = axes.ravel()

for i in range(min(num_samples, len(misclassified_images))):
    img = misclassified_images[i].squeeze().numpy()
    
    # Color based on prediction correctness (red for incorrect)
    color = 'red' if misclassified_true[i] != misclassified_pred[i] else 'cyan'
    props = dict(boxstyle='round', facecolor=color, alpha=0.5)
    
    axes[i].imshow(img, cmap=plt.cm.binary)
    axes[i].set_title(class_names[misclassified_true[i].item()], fontsize=10)
    axes[i].text(0.1, 0.95, class_names[misclassified_pred[i].item()],
                 transform=axes[i].transAxes, fontsize=10,
                 verticalalignment='top', bbox=props)
    axes[i].axis('off')